# Task 4 — Model Integration, Evaluation & System Simulation
**Gaju — Student D — Formative 2: Multimodal Data Preprocessing Assignment**

This notebook is the integration layer that ties together:
- **Andrew's** merged tabular dataset (`data/processed/merged_dataset.csv`)
- **Divine's** facial recognition model (mocked here until her real model lands)
- **HonourGod's** voice verification model (already implemented and used as-is)
- **Gaju's own** product recommendation model (trained in this notebook)

into a single CLI state machine: **Face check → Product prediction (silent) → Voice check → Reveal product**.

> **Status note:** `facial_recognition_model.py` and `image_preprocessing.py`
> are still TODO stubs from Divine as of this notebook. `scripts/mock_facial_recognition_model.py`
> stands in with the *exact same function signature* so integration work isn't blocked.
> Swap `MOCK_MODE = False` in `app/cli_app.py` once her real model is delivered — nothing
> else in this notebook needs to change.

## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import json
import pandas as pd

pd.set_option('display.max_columns', None)

## 1. Generate/confirm the data each model needs

If you already have real `merged_dataset.csv` from Andrew and real raw audio
from HonourGod, skip this cell and just make sure they're in `data/processed/`
and `data/raw/audio/`. This mock generator is only here so the pipeline is
testable before all deliverables arrive.

In [2]:
from scripts.generate_mock_data import main as generate_mock_data
# Comment this out once real team data is in place
generate_mock_data()

Wrote mock merged_dataset.csv: (120, 17)
Wrote mock raw audio to /home/claude/project/data/raw/audio


## 2. Run/confirm each pipeline stage

In [3]:
# HonourGod's audio pipeline (already implemented)
from scripts.audio_preprocessing import process_audio_directory
process_audio_directory()
pd.read_csv('data/processed/audio_features.csv').head()

,member,sample_name,phrase,mfcc_mean,mfcc_std,spectral_centroid_mean,spectral_rolloff_mean,zero_crossing_rate_mean,rms_mean,duration_sec
0,andrew,confirm_transaction,confirm_transaction,-4.943178,61.948074,1771.140451,5119.140625,0.017105,0.207232,1.000000
1,andrew,confirm_transaction_pitch_shift,confirm_transaction,-8.237418,72.966873,1381.189927,4240.722656,0.016830,0.188596,1.000000
2,andrew,confirm_transaction_time_stretch,confirm_transaction,-5.803399,68.489174,1531.506238,4609.913793,0.015524,0.205533,0.909062
3,andrew,confirm_transaction_background_noise,confirm_transaction,-3.382249,53.219693,2115.129861,5639.160156,0.020828,0.207437,1.000000
4,andrew,yes_approve,yes_approve,-4.515448,61.992695,1770.040867,5140.136719,0.017929,0.207235,1.000000


In [4]:
# HonourGod's voice verification model (already implemented)
from scripts.voice_verification_model import train_voice_verification_model
voice_report = train_voice_verification_model()
print(json.dumps(voice_report['selected_model_metrics'], indent=2))

{
  "accuracy": 0.94,
  "f1_score": 0.8235294117647058,
  "log_loss": 0.20503515122690227,
  "confusion_matrix": [
    [
      80,
      0
    ],
    [
      6,
      14
    ]
  ]
}


In [5]:
# Gaju's own product recommendation model (built for this task)
from scripts.product_recommendation_model import train_product_recommendation_model
product_report = train_product_recommendation_model()
print(json.dumps(product_report['selected_model_metrics'], indent=2))

{
  "accuracy": 0.20833333333333334,
  "f1_score": 0.22182539682539684,
  "log_loss": 1.97499142180442
}


## 3. Model comparison table

Pulls together accuracy / F1 / loss from every model's JSON report — this
table goes straight into the evaluation section of the report.

In [6]:
rows = []
for name, path in [
    ('Voice Verification', 'data/processed/voice_verification_model.json'),
    ('Product Recommendation', 'data/processed/product_recommendation_model.json'),
    # ('Facial Recognition', 'data/processed/facial_recognition_model.json'),  # add once Divine delivers
]:
    with open(path) as f:
        report = json.load(f)
    m = report['selected_model_metrics']
    rows.append({
        'model': name,
        'selected_algorithm': report['selected_model'],
        'accuracy': round(m.get('accuracy', float('nan')), 3),
        'f1_score': round(m.get('f1_score', float('nan')), 3),
        'log_loss': round(m.get('log_loss', float('nan')), 3),
    })

pd.DataFrame(rows)

,model,selected_algorithm,accuracy,f1_score,log_loss
0,Voice Verification,random_forest,0.940,0.824,0.205
1,Product Recommendation,gradient_boosting,0.208,0.222,1.975


## 4. Run the CLI state machine interactively

Uncomment to try it live from the notebook (prompts for image/customer_id/audio paths).

In [7]:
# from app.cli_app import interactive_main
# interactive_main()

## 5. Required simulation scenarios (Section 6 of the brief)

1. Unauthorized image attempt (fail at Stage 1)
2. Authorized face, unauthorized voice (fail at Stage 3)
3. Full successful end-to-end path (reach Stage 4)

In [8]:
from app.cli_app import run_transaction

AUDIO_DIR = 'data/raw/audio'

scenarios = [
    ('Unauthorized image attempt', 'unauthorized_face_01.jpg', 1, f'{AUDIO_DIR}/andrew/yes_approve.wav', 'denied'),
    ('Authorized face, unauthorized voice', 'andrew_neutral.jpg', 1, f'{AUDIO_DIR}/unauthorized/yes_approve.wav', 'denied'),
    ('Full successful end-to-end path', 'andrew_neutral.jpg', 1, f'{AUDIO_DIR}/andrew/yes_approve.wav', 'granted'),
]

results = []
for name, image_path, customer_id, audio_path, expected in scenarios:
    print(f"\n----- Scenario: {name} -----")
    outcome = run_transaction(image_path, customer_id, audio_path)
    results.append({'scenario': name, 'expected': expected, 'got': outcome['status'],
                     'passed': outcome['status'] == expected})

pd.DataFrame(results)


----- Scenario: Unauthorized image attempt -----
[Stage 1] Face check on 'unauthorized_face_01.jpg' -> authorized=False, member=None, confidence=0.12

*** ACCESS DENIED at Stage 1 (Face): unrecognized face (confidence=0.12) ***


----- Scenario: Authorized face, unauthorized voice -----
[Stage 1] Face check on 'andrew_neutral.jpg' -> authorized=True, member=andrew, confidence=0.94
[Stage 2] Product prediction computed silently (withheld until Stage 4).
[Stage 3] Voice check for claimed identity 'andrew' -> verified=False, confidence=0.00

*** ACCESS DENIED at Stage 3 (Voice): voice did not match (confidence=0.00) ***


----- Scenario: Full successful end-to-end path -----
[Stage 1] Face check on 'andrew_neutral.jpg' -> authorized=True, member=andrew, confidence=0.94
[Stage 2] Product prediction computed silently (withheld until Stage 4).


[Stage 3] Voice check for claimed identity 'andrew' -> verified=True, confidence=0.89
[Stage 4] ACCESS GRANTED -> Recommended product: Electronics (confidence=0.97)


,scenario,expected,got,passed
0,Unauthorized image attempt,denied,denied,True
1,"Authorized face, unauthorized voice",denied,denied,True
2,Full successful end-to-end path,granted,granted,True


## 6. Next steps / handoff notes

- [ ] Replace `data/processed/merged_dataset.csv` with Andrew's real output
- [ ] Replace `data/raw/audio/` with HonourGod's real recordings, re-run Section 2
- [ ] Once Divine delivers `facial_recognition_model.py` + `image_features.csv`:
      set `MOCK_MODE = False` in `app/cli_app.py`, re-run Section 5
- [ ] Paste the Section 3 comparison table and Section 5 results table into the report
- [ ] Record the final simulation video using the real (non-mock) run